In [59]:
import os
import time
import json
import gc
import random

import numpy as np
import polars as pl

# Machine Learning
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import joblib

# Tree Models
import xgboost as xgb
import lightgbm as lgb

# Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras import regularizers

In [52]:
event_path = r"Data\BORG\instance_events-*.json.gz"
usage_path = r"Data\BORG\instance_usage-*.json.gz"

In [53]:
# Quick debug to see the actual 2019 Usage schema
usage_sample = pl.read_ndjson(usage_path, n_rows=5)
print(usage_sample.columns)
print(usage_sample.head(1))
event_sample = pl.read_ndjson(event_path, n_rows=5)
print(event_sample.columns)
print(event_sample.head(1))

['start_time', 'end_time', 'collection_id', 'instance_index', 'machine_id', 'alloc_collection_id', 'alloc_instance_index', 'collection_type', 'average_usage', 'maximum_usage', 'random_sample_usage', 'assigned_memory', 'page_cache_memory', 'cycles_per_instruction', 'memory_accesses_per_instruction', 'sample_rate', 'cpu_usage_distribution', 'tail_cpu_usage_distribution']
shape: (1, 18)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ start_tim ┆ end_time  ┆ collectio ┆ instance_ ┆ … ┆ memory_ac ┆ sample_ra ┆ cpu_usage ┆ tail_cpu │
│ e         ┆ ---       ┆ n_id      ┆ index     ┆   ┆ cesses_pe ┆ te        ┆ _distribu ┆ _usage_d │
│ ---       ┆ str       ┆ ---       ┆ ---       ┆   ┆ r_instruc ┆ ---       ┆ tion      ┆ istribut │
│ str       ┆           ┆ str       ┆ str       ┆   ┆ tio…      ┆ f64       ┆ ---       ┆ ion      │
│           ┆           ┆           ┆           ┆   ┆ ---       ┆           ┆ list[f64] ┆ ---      │
│      

In [54]:
### print("1. Loading raw data with labels [4, 5, 7] and 8 Features...")
event_schema = {
    "machine_id": pl.String, "time": pl.String, "type": pl.String,
    "priority": pl.String, "missing_type": pl.String,
    "resource_request": pl.Struct([pl.Field("cpus", pl.Float64), pl.Field("memory", pl.Float64)]),
    "constraint": pl.List(pl.Struct([])) 
}
usage_schema = {
    "machine_id": pl.String, "start_time": pl.String,
    "average_usage": pl.Struct([pl.Field("cpus", pl.Float64), pl.Field("memory", pl.Float64)]),
    "maximum_usage": pl.Struct([pl.Field("cpus", pl.Float64), pl.Field("memory", pl.Float64)]),
    "random_sample_usage": pl.Struct([pl.Field("cpus", pl.Float64)]),
    "assigned_memory": pl.Float64, "page_cache_memory": pl.Float64, 
    "cycles_per_instruction": pl.Float64, "memory_accesses_per_instruction": pl.Float64,
    "sample_rate": pl.Float64,
    "cpu_usage_distribution": pl.List(pl.Float64),
    "tail_cpu_usage_distribution": pl.List(pl.Float64)
}

print("Loading and Parsing Raw JSONs...")
failure_labels = [4, 5, 7]

# Load Events (Priority, Demands, Missing Types, Constraints)
events = (pl.scan_ndjson(event_path, schema=event_schema)
          .select([
              pl.col("machine_id").cast(pl.Float64), 
              pl.col("time").cast(pl.Float64),
              pl.col("type").cast(pl.Int32).is_in(failure_labels).cast(pl.Int8).alias("error_signal"),
              pl.col("priority").cast(pl.Float64).alias("ev_priority"),
              pl.col("missing_type").cast(pl.Float64).alias("ev_missing_type"),
              pl.col("resource_request").struct.field("cpus").alias("ev_req_cpu"),
              pl.col("resource_request").struct.field("memory").alias("ev_req_mem"),
              pl.col("constraint").list.len().cast(pl.Float64).alias("ev_num_constraints")
          ])
          .sort("time"))

# Load Usage (Sample Rate, MAPI, Cache, Averages, Maxes, Distributions)
usage = (pl.scan_ndjson(usage_path, schema=usage_schema)
         .select([
             pl.col("machine_id").cast(pl.Float64), 
             pl.col("start_time").cast(pl.Float64).alias("time"),
             pl.col("sample_rate").alias("us_sample_rate"),
             pl.col("page_cache_memory").alias("us_page_cache_mem"),
             pl.col("memory_accesses_per_instruction").alias("us_mapi"),
             pl.col("average_usage").struct.field("cpus").alias("us_avg_cpu"),
             pl.col("average_usage").struct.field("memory").alias("us_avg_mem"),
             pl.col("maximum_usage").struct.field("cpus").alias("us_max_cpu"),
             pl.col("maximum_usage").struct.field("memory").alias("us_max_mem"),
             pl.col("cpu_usage_distribution").list.last().cast(pl.Float64).alias("us_cpu_dist_tail"),
             pl.col("random_sample_usage").struct.field("cpus").alias("us_rand_cpu"),
             pl.col("cycles_per_instruction").alias("us_cpi"),
             pl.col("assigned_memory").alias("us_assigned_mem"),
             pl.col("tail_cpu_usage_distribution").list.last().cast(pl.Float64).alias("us_cpu_tail_dist_tail")
         ])
         .sort("time"))

df_raw = usage.join_asof(events, on="time", by="machine_id", strategy="backward").collect()

print("2. Resampling to strict 5-minute time grid...")
# Expanded to 13 most important Features
ev_features = [
    'ev_req_cpu', 
    'ev_priority', 
    'ev_num_constraints',
    'ev_req_mem'
]

us_features = [
    'us_assigned_mem', 
    'us_cpi', 
    'us_page_cache_mem', 
    'us_avg_mem', 
    'us_avg_cpu', 
    'us_mapi', 
    'us_sample_rate',
    'us_cpu_dist_tail',
    'us_max_cpu'
]

features = ev_features + us_features


features = ev_features + us_features

df_grid = df_raw.with_columns(
    (pl.col("time") // 300_000_000).cast(pl.Int64).alias("time_bucket")
)

# Dynamically aggregate all 9 features
df_grid = df_grid.group_by(["machine_id", "time_bucket"]).agg(
    [pl.col("error_signal").max()] + [pl.col(f).mean() for f in features]
).sort(["machine_id", "time_bucket"]).fill_null(0.0)

print("3. Performing per-machine temporal split (70% train / 30% test)...")
train_rows, test_rows = [], []
for _, group in df_grid.group_by("machine_id"):
    group = group.sort("time_bucket")
    cut = int(len(group) * 0.70)
    train_rows.append(group.slice(0, cut))
    test_rows.append(group.slice(cut))
df_train_grid = pl.concat(train_rows)
df_test_grid = pl.concat(test_rows)

del df_raw, df_grid
gc.collect()

print("4. Applying StandardScaler...")
scaler = RobustScaler()

df_train_scaled_vals = scaler.fit_transform(df_train_grid.select(features).to_numpy())
df_train_grid = df_train_grid.with_columns([
    pl.Series(name=features[i], values=df_train_scaled_vals[:, i]) for i in range(len(features))
])

df_test_scaled_vals = scaler.transform(df_test_grid.select(features).to_numpy())
df_test_grid = df_test_grid.with_columns([
    pl.Series(name=features[i], values=df_test_scaled_vals[:, i]) for i in range(len(features))
])

print("5. Extracting sequences (with contiguity gap checks)...")
def extract_sequences_from_df(df, window_size=60):
    groups = df.group_by("machine_id")
    X, y = [], []
    MAX_GAP = 24
    
    for _, group in groups:
        buckets = group.select("time_bucket").to_numpy().flatten()
        if len(buckets) == 0: 
            continue
            
        vals = group.select(features).to_numpy()
        signals = group.select("error_signal").to_numpy().flatten()
        
        chunks = []
        current_chunk_buckets = [buckets[0]]
        current_chunk_vals = [vals[0]]
        current_chunk_signals = [signals[0]]
        
        for i in range(1, len(buckets)):
            gap = buckets[i] - buckets[i-1]
            if gap > MAX_GAP:
                chunks.append((current_chunk_buckets, current_chunk_vals, current_chunk_signals))
                current_chunk_buckets = [buckets[i]]
                current_chunk_vals = [vals[i]]
                current_chunk_signals = [signals[i]]
            else:
                current_chunk_buckets.append(buckets[i])
                current_chunk_vals.append(vals[i])
                current_chunk_signals.append(signals[i])
                
        chunks.append((current_chunk_buckets, current_chunk_vals, current_chunk_signals))
        
        for c_buckets, c_vals, c_signals in chunks:
            if len(c_buckets) == 0: continue
            min_b = int(c_buckets[0])
            max_b = int(c_buckets[-1])
            
            if max_b - min_b + 1 < window_size + 6:
                continue
                
            length = max_b - min_b + 1
            if length > 100000: 
                continue
                
            dense_vals = np.zeros((length, len(features)), dtype='float32')
            dense_signals = np.zeros(length, dtype='int8')
            
            current_val = c_vals[0]
            idx = 0
            for b in range(length):
                actual_bucket = min_b + b
                if idx < len(c_buckets) and c_buckets[idx] == actual_bucket:
                    current_val = c_vals[idx]
                    dense_signals[b] = c_signals[idx]
                    idx += 1
                dense_vals[b] = current_val
                
            for T in range(window_size, length - 12):
                if dense_signals[T - window_size : T].sum() == 0:
                    X.append(dense_vals[T - window_size : T])
                    y.append(1 if dense_signals[T : T + 12].sum() > 0 else 0)
                
                
    return np.array(X), np.array(y)


def undersample_to_ratio(X, y, target_healthy_ratio=0.50):
    """
    Reduces the majority class (Healthy, 0) to hit a specific ratio against 
    the minority class (Failure, 1), preserving 100% of failure sequences.
    """
    fail_idx = np.where(y == 1)[0]
    healthy_idx = np.where(y == 0)[0]
    
    # Calculate required healthy count (Failures * 1.5 for a 50:50 split)
    target_failure_ratio = 1.0 - target_healthy_ratio
    multiplier = target_healthy_ratio / target_failure_ratio
    target_healthy_count = int(len(fail_idx) * multiplier)
    
    # Safety catch: don't request more than we have
    target_healthy_count = min(target_healthy_count, len(healthy_idx))
    
    # Randomly sample the healthy indices
    rng = np.random.default_rng(42)
    sampled_healthy_idx = rng.choice(healthy_idx, size=target_healthy_count, replace=False)
    
    # Combine and shuffle
    keep_idx = np.concatenate([fail_idx, sampled_healthy_idx])
    rng.shuffle(keep_idx)
    
    return X[keep_idx], y[keep_idx]

window_size = 24
X_train_raw, y_train_raw = extract_sequences_from_df(df_train_grid, window_size)
X_test_raw, y_test_raw   = extract_sequences_from_df(df_test_grid, window_size)

print(f"Original Train RAM footprint: {X_train_raw.nbytes / 1e9:.2f} GB")
print(f"Original Train Positive Rate: {y_train_raw.mean():.4f}")

cut = int(len(X_train_raw) * 0.9)
X_train_imbalanced, X_val = X_train_raw[:cut], X_train_raw[cut:]
y_train_imbalanced, y_val = y_train_raw[:cut], y_train_raw[cut:]

# 2. Apply the 50:50 Undersampling (Train Set Only)
X_train_60, y_train_60 = undersample_to_ratio(X_train_imbalanced, y_train_imbalanced, target_healthy_ratio=0.50)
X_test_60, y_test_60   = X_test_raw, y_test_raw  

print(f"\nBalanced Train RAM footprint: {X_train_60.nbytes / 1e9:.2f} GB")
print(f"Balanced Train Positive Rate: {y_train_60.mean():.4f}")
print(f"Balanced Train Shapes: X={X_train_60.shape}, y={y_train_60.shape}")

# Clean up raw arrays to immediately free RAM
del X_train_raw, y_train_raw, X_train_imbalanced, y_train_imbalanced
import gc
gc.collect()


Loading and Parsing Raw JSONs...


C:\Users\sabbu\AppData\Local\Temp\ipykernel_28516\797902719.py:57: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_raw = usage.join_asof(events, on="time", by="machine_id", strategy="backward").collect()


2. Resampling to strict 5-minute time grid...
3. Performing per-machine temporal split (70% train / 30% test)...
4. Applying StandardScaler...
5. Extracting sequences (with contiguity gap checks)...
Original Train RAM footprint: 0.16 GB
Original Train Positive Rate: 0.3284

Balanced Train RAM footprint: 0.09 GB
Balanced Train Positive Rate: 0.5000
Balanced Train Shapes: X=(75020, 24, 13), y=(75020,)


0

In [55]:
def engineer_tree_features(X_3d):
    # X_3d shape: (N, steps, num_features)
    # 1. Endpoint value
    last_step = X_3d[:, -1, :]
    # 2. Sequence mean
    mean_val = np.mean(X_3d, axis=1)
    # 3. Trajectory velocity (delta between start and end of window)
    delta_val = X_3d[:, -1, :] - X_3d[:, 0, :]
    # 4. Standard deviation (instability / volatility metric)
    std_val = np.std(X_3d, axis=1)
    # 5. Peak load
    max_val = np.max(X_3d, axis=1)
    
    return np.hstack([last_step, mean_val, delta_val, std_val, max_val])

X_train_tree = engineer_tree_features(X_train_60)
X_test_tree  = engineer_tree_features(X_test_60)
print("hello")

hello


In [60]:
def build_CNN_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(64, kernel_size=3, activation='relu'),
        layers.Dropout(0.2),
        layers.Flatten(),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

def build_LSTM_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

def build_Hybrid_model(input_shape):
    """
    Upgraded Hybrid Model (Pure Accuracy).
    Uses deeper CNN feature extraction, a Bidirectional LSTM, 
    and Multi-Head Attention to capture complex temporal degradation patterns.
    """
    inputs = layers.Input(shape=input_shape)
    
    # 1. Feature Extraction (1D CNN)
    x = layers.Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    # Reduce sequence length slightly to help the LSTM focus on higher-level features
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    
    # 2. Sequence Modeling (Bidirectional LSTM)
    # Bidirectional processing looks at the sequence in both directions
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Dropout(0.2)(x)
    
    # 3. Attention Mechanism (Multi-Head)
    # Vastly superior to standard Attention() for learning complex time dependencies
    attention_output = layers.MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attention_output)
    
    # 4. Pooling & Output
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

def build_GRU_model(input_shape):
    """
    GRU (Gated Recurrent Unit) is similar to LSTM but has fewer parameters. 
    It trains faster, uses less memory, and often achieves identical accuracy.
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.GRU(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model
    
def build_ResNet1D_model(input_shape):
    """
    1D-ResNet: Now heavily regularized with L2 weight decay and Dropout(0.5)
    """
    inputs = layers.Input(shape=input_shape)
    
    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu', 
                      kernel_regularizer=regularizers.l2(0.001))(inputs)
    x = layers.BatchNormalization()(x)
    
    # Residual Block 1
    res = x
    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Conv1D(64, kernel_size=3, padding='same', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([res, x])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    
    # Residual Block 2
    res = layers.Conv1D(128, kernel_size=1, padding='same')(x)
    x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x) # Increased Dropout
    x = layers.Conv1D(128, kernel_size=3, padding='same', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([res, x])
    x = layers.Activation('relu')(x)
    
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.Dropout(0.5)(x) # Heavy Dropout on the dense layer
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model


def build_TCN_model(input_shape):
    """
    TCN: Now heavily regularized with L2 weight decay and Dropout(0.4)
    """
    inputs = layers.Input(shape=input_shape)
    x = inputs
    
    for dilation_rate in [1, 2, 4, 8]:
        res = layers.Conv1D(64, kernel_size=1, padding='same')(x)
        
        x = layers.Conv1D(64, kernel_size=3, padding='causal', dilation_rate=dilation_rate, 
                          activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.4)(x) # Increased Dropout
        x = layers.Conv1D(64, kernel_size=3, padding='causal', dilation_rate=dilation_rate, 
                          kernel_regularizer=regularizers.l2(0.001))(x)
        x = layers.BatchNormalization()(x)
        
        x = layers.Add()([res, x])
        x = layers.Activation('relu')(x)
        
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model


def build_ResNet_Hybrid_model(input_shape):
    """
    1D-ResNet Hybrid: Replaces the basic CNN feature extractor with a 
    powerful 1D-ResNet block, followed by Bidirectional LSTM and Attention.
    """    
    inputs = layers.Input(shape=input_shape)
    
    # 1. ResNet Feature Extraction Block
    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    
    # block 1
    res = x
    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, kernel_size=3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([res, x])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # block 2
    res = layers.Conv1D(128, kernel_size=1, padding='same')(x)
    x = layers.Conv1D(128, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, kernel_size=3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([res, x])
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    
    # 2. Sequence Modeling
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Dropout(0.2)(x)
    
    # 3. Attention
    attention_output = layers.MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attention_output)
    
    # 4. Pooling & Output
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

def build_Deep_CNN_model(input_shape):
    """
    4-Layer Deep CNN: Stacks 4 convolutional layers to learn complex 
    hierarchical features while maintaining the blazing fast speed of a CNN.
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # Layer 1
        layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        
        # Layer 2
        layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2), # Reduces the timeline length to speed up next layers
        
        # Layer 3 (Increase filters to learn more complex patterns)
        layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        
        # Layer 4
        layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        
        # Compress the sequence into a single vector
        layers.GlobalAveragePooling1D(),
        
        # Final classification
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model


In [ ]:
models_to_test = {
    # Commented out so they don't run again, saving you time!
    "XGBoost": "baseline_supervised",
    "LightGBM": "baseline_supervised",
    "Individual CNN": build_CNN_model,
    "1D-ResNet": build_ResNet1D_model,
    "TCN": build_TCN_model,
    "Individual LSTM": build_LSTM_model,
    "Individual GRU": build_GRU_model,
    # "Hybrid": build_Hybrid_model,
    # "Deep CNN": build_Deep_CNN_model,
    # "ResNet-Hybrid": build_ResNet_Hybrid_model
}

results_file = "model_results_borg.json"

# Load existing results as to not overwrite any
results = {}
if os.path.exists(results_file):
    with open(results_file, 'r') as f:
        try:
            results = json.load(f)
            print(f"Loaded {len(results)} existing model scores from {results_file}")
        except json.JSONDecodeError:
            pass
            
for name, factory in models_to_test.items():
    print(f"\n--- Training {name} ---")
    
    start_time = time.time()
    
    if isinstance(factory, str):
        if factory == "baseline_supervised":
            if name == "XGBoost":
                m = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
            elif name == "LightGBM":
                m = lgb.LGBMClassifier(random_state=42)
            else:
                raise ValueError(f"Unknown baseline: {name}")
            
            # Explicitly use the 60-step engineered features
            m.fit(X_train_tree, y_train_60)
            
            if name in ["XGBoost", "LightGBM"]:
                y_prob = m.predict_proba(X_test_tree)[:, 1]
                    
                if name == "XGBoost":
                    m.save_model("fullscale_xgb_model_457.json")
                    print("Saved XGBoost model to 'fullscale_xgb_model_457.json'")
                elif name == "LightGBM":
                    joblib.dump(m, 'fullscale_lgb_model_457.pkl')
                    print("Saved LightGBM model to 'fullscale_lgb_model_457.pkl'")
                
                y_pred = (y_prob > 0.4).astype(int)
            else:
                raise ValueError(f"Unknown unsupervised baseline: {name}")

    else:
        print(f"  -> Training on {len(X_train_60)} perfectly balanced sequences...")
        
        # Explicitly build model using X_train_60 shape
        m = factory(X_train_60.shape[1:])
        
        # Instantiate a fresh callback per model
        fresh_early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
        
        # Pass the fresh callback
        m.fit(X_train_60, y_train_60, epochs=10, batch_size=512, verbose=1, validation_data=(X_val, y_val), callbacks=[fresh_early_stop])

        # Explicitly predict using X_test_60
        y_prob = m.predict(X_test_60, batch_size=1024).flatten()
        y_pred = (y_prob > 0.5).astype(int)
        
        save_filename = f"{name.replace(' ', '_').replace('-', '_')}_model.keras"
        m.save(save_filename)
        print(f"Saved {name} model to '{save_filename}'")
        
    end_time = time.time()
    training_duration = end_time - start_time
    
    score = f1_score(y_test_60, y_pred)
    
    # Save/Append both metrics safely
    results[name] = {
        "f1_score": float(score),
        "training_time_seconds": round(training_duration, 2)
    }
    
    print(f"{name} F1 Score on Test Set: {score:.4f} (Took {training_duration:.2f} seconds)")
    
    if not isinstance(factory, str):
        del m
        tf.keras.backend.clear_session()
        gc.collect()

# Save appended results back to the file
with open(results_file, 'w') as f:
    json.dump(results, f, indent=4)
print("\n" + "="*40)
print(f"RESULTS SAFELY APPENDED TO: {os.path.abspath(results_file)}")
print("="*40)

Loaded 10 existing model scores from model_results.json

--- Training XGBoost ---


C:\Users\sabbu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\training.py:183: UserWarning: [02:16:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Saved XGBoost model to 'fullscale_xgb_model_457.json'
XGBoost F1 Score on Test Set: 0.5370 (Took 0.75 seconds)

--- Training LightGBM ---
[LightGBM] [Info] Number of positive: 37510, number of negative: 37510
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011136 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14590
[LightGBM] [Info] Number of data points in the train set: 75020, number of used features: 65
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Saved LightGBM model to 'fullscale_lgb_model_457.pkl'
LightGBM F1 Score on Test Set: 0.5484 (Took 0.77 seconds)

--- Training Individual CNN ---
  -> Training on 75020 perfectly balanced sequences...
Epoch 1/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - AUC: 0.6654 - loss: 0.6503 - val_AUC: 0.6899 - val_loss: 0.6266
Epoch 2/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - AUC: 0.7084 - loss: 0.6239 - val_AUC: 0.6958 - val